# Ball Balance — Training
PPO via rsl-rl-lib >= 5. Unitree G1 fixed-base humanoid.  
Runs on Kaggle GPU or locally for testing.

In [18]:
# ── 1. Per-run config ─────────────────────────────────────────────────────
import os

ON_KAGGLE = os.path.exists('/kaggle')

BRANCH         = 'goal_randomization'
TASK           = 'BallBalance-DualArm-v0'   # any registered gym env ID
EXP_NAME       = 'BallBalance-DualArm-v0'
N_ENVS         = 2048 if ON_KAGGLE else 4
MAX_ITERATIONS = 1000 if ON_KAGGLE else 5
ACTION_DELTA   = 0.3

REPO_DIR = '/kaggle/working/bimanual-project' if ON_KAGGLE else os.path.abspath('.')
# LOG_DIR is set after training since the timestamp is generated by train.py

print(f'ON_KAGGLE: {ON_KAGGLE}')
print(f'TASK:      {TASK}')
print(f'REPO_DIR:  {REPO_DIR}')


ON_KAGGLE: False
TASK:      BallBalance-DualArm-v0
REPO_DIR:  /home/kenan/Code/UCSD/CSE190/bimanual-project


In [2]:
# ── 2. Clone / pull repo  (Kaggle only) ───────────────────────────────────
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    import subprocess

    token = UserSecretsClient().get_secret('GITHUB_TOKEN')

    if not os.path.exists(REPO_DIR):
        result = subprocess.run([
            'git', 'clone', '--branch', BRANCH,
            f'https://{token}@github.com/sharana-sabesan09/bimanual-project.git',
            REPO_DIR,
        ], capture_output=True, text=True)
        print(result.stdout or result.stderr)
    else:
        result = subprocess.run(
            ['git', '-C', REPO_DIR, 'pull'],
            capture_output=True, text=True
        )
        print(result.stdout)

    commit = subprocess.run(
        ['git', '-C', REPO_DIR, 'log', '-1', '--pretty=%h %s'],
        capture_output=True, text=True
    )
    print('Commit:', commit.stdout.strip())
else:
    import subprocess
    commit = subprocess.run(
        ['git', 'log', '-1', '--pretty=%h %s'],
        capture_output=True, text=True
    )
    print('Local repo — commit:', commit.stdout.strip())

Local repo — commit: d7f4842 goal_marker


In [3]:
# ── 3. Install dependencies  (Kaggle only) ────────────────────────────────
if ON_KAGGLE:
    import subprocess
    subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
    print('Dependencies installed.')
else:
    print('Local — skipping installs.')

Local — skipping installs.


In [4]:
# ── 4. Setup paths ────────────────────────────────────────────────────────
import sys

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
os.makedirs(LOG_DIR, exist_ok=True)

print(f'Working dir: {os.getcwd()}')
print(f'Log dir:     {LOG_DIR}')

NameError: name 'LOG_DIR' is not defined

In [5]:
# ── 5. Verify GPU ─────────────────────────────────────────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU:  {props.name}')
    print(f'VRAM: {props.total_memory / 1e9:.1f} GB')
else:
    print('No GPU — training will use CPU (fine for local testing, slow for real runs)')

CUDA available: False
No GPU — training will use CPU (fine for local testing, slow for real runs)


/home/kenan/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [19]:
# ── 7. Train ──────────────────────────────────────────────────────────────
cmd = (
    f"python scripts/rsl_rl/train.py"
    f" --task {TASK}"
    f" -e {EXP_NAME}"
    f" -n {N_ENVS}"
    f" --max_iterations {MAX_ITERATIONS}"
    f" --action_delta {ACTION_DELTA}"
    f" --headless"
)

print(f'Running: {cmd}')
!{cmd}

# pick up the timestamped run dir created by train.py
import glob
runs = sorted(glob.glob(os.path.join(REPO_DIR, "logs", EXP_NAME, "*")))
LOG_DIR = runs[-1]  # most recent
print(f'Log dir: {LOG_DIR}')


Running: python scripts/rsl_rl/train.py --task BallBalance-DualArm-v0 -e BallBalance-DualArm-v0 -n 4 --max_iterations 5 --action_delta 0.3 --headless
Initializing Genesis with GPU backend...
/home/kenan/Code/UCSD/CSE190/bimanual-project/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
[Genesis] [22:14:09] [WARNING] Backend gs.gpu not available on this machine. Falling back to CPU.
[Genesis] [22:14:18] [WARNING] Link 'left_ankle_roll_link' has dubious mass 0.608 compared to the estimate from geometry 0.003 

In [20]:
os.path.join(REPO_DIR, "logs", EXP_NAME, "*")

'/home/kenan/Code/UCSD/CSE190/bimanual-project/logs/BallBalance-DualArm-v0/*'

In [ ]:
os.path.join(REPO_DIR, "logs", EXP_NAME, "*")

In [ ]:
# ── 8. Plot training curves ───────────────────────────────────────────────
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

ea = EventAccumulator(LOG_DIR)
ea.Reload()

def extract(tag):
    events = ea.Scalars(tag)
    return [e.step for e in events], [e.value for e in events]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

steps, vals = extract('Train/mean_reward')
axes[0].plot(steps, vals)
axes[0].set_title('Mean Reward')
axes[0].set_xlabel('Iteration')
axes[0].grid(True)

steps, vals = extract('Train/mean_episode_length')
axes[1].plot(steps, vals, color='orange')
axes[1].axhline(500, color='gray', linestyle='--', label='max ep length')
axes[1].set_title('Mean Episode Length')
axes[1].set_xlabel('Iteration')
axes[1].legend()
axes[1].grid(True)

plt.suptitle(f'{EXP_NAME}  ({TASK})', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'training_curves.png'), dpi=150)
plt.show()

_, rewards = extract('Train/mean_reward')
_, ep_lens = extract('Train/mean_episode_length')
print(f'Final reward: {rewards[-1]:.2f}  (best {max(rewards):.2f})')
print(f'Final ep len: {ep_lens[-1]:.1f}  (best {max(ep_lens):.1f} / 500)')

In [ ]:
# ── 9. Confirm outputs ────────────────────────────────────────────────────
print(f'Files in {LOG_DIR}:')
for f in sorted(os.listdir(LOG_DIR)):
    size = os.path.getsize(os.path.join(LOG_DIR, f)) / 1e6
    print(f'  {f}  ({size:.1f} MB)')